In [2]:
# ! uv pip install litellm

In [3]:
import os
from litellm import completion
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
	raise ValueError("OPENROUTER_API_KEY is not set in environment or .env file")
os.environ["OPENROUTER_API_KEY"] = api_key

# Optional OpenRouter headers
# os.environ["OR_SITE_URL"] = "https://localhost"
# os.environ["OR_APP_NAME"] = "litellm-learning"

In [5]:
from typing import cast
from litellm.types.utils import ModelResponse

response = cast(
    ModelResponse,
    completion(
        model="openrouter/stealth/ox-alpha",
        messages=[
            {"role": "user", "content": "How many r's are in the word 'strawberry'?"}
        ],
        # Enable reasoning (OpenRouter-specific)
        reasoning={"enabled": True},
        max_tokens=500,
        # stream=False,
    ),
)

# Main answer
print(response.choices[0].message.content)

# Reasoning details (if the model returns them)
msg = response.choices[0].message
print(msg)

# Some models put reasoning in provider_specific_fields or reasoning_content
if hasattr(msg, "reasoning_content") and msg.reasoning_content:
    print(msg.reasoning_content)


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

There are **3** r's in the word "strawberry":

**st r awbe rr y**

- 1st r: st**r**awberry
- 2nd and 3rd r's: strawbe**rr**y
Provider List: https://docs.litellm.ai/docs/providers


Message(content='There are **3** r\'s in the word "strawberry":\n\n**st r awbe rr y**\n\n- 1st r: st**r**awberry\n- 2nd and 3rd r\'s: strawbe**rr**y', role='assistant', tool_calls=None, function_call=None, reasoning_content='The user is asking how many r\'s are in the word \'strawberry\'.\n\nLet me spell it out: s-t-r-a-w-b-e-r-r-y\n\ns (1)\nt (2)\nr (3) - first r\na (4)\nw (5)\nb (6)\ne (7)\nr (8) - second r\nr (9) - third r\ny (10)\n\nSo there are 3 r\'s in "strawberry".', provider_specific_fields={'refusal': None, 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking how many r\'s are in the word \'strawberry\'.\n\nLet me spell it out: s-t-r-a-w-b-e-r-r-y\n\ns (1)\nt (2)\n

In [6]:
# 3 · Multi-turn — preserve reasoning_details across turns

messages = cast(
    list[dict[str, object]],
    [{"role": "user", "content": "How many r's are in the word 'strawberry'?"}],
)

resp1 = cast(
    ModelResponse,
    completion(
        model="openrouter/stealth/ox-alpha",
        messages=messages,
        reasoning={"enabled": True},
        max_tokens=800,
    ),
)

assistant_msg = resp1.choices[0].message

print(f"\n{'━'*60}")
print(f"  Turn 1 — First answer")
print(f"{'━'*60}")
print(f"[answer]\n{assistant_msg.content}")

# Preserve full assistant message (including any reasoning fields)
extra_fields = {
    k: v
    for k, v in assistant_msg.__dict__.items()
    if k not in ("role", "content", "tool_calls", "function_call") and v is not None
}
messages.append(
    {
        "role": "assistant",
        "content": assistant_msg.content or "",
        **extra_fields,
    }
)

# ── Turn 2
messages.append(
    {
        "role": "user",
        "content": "Can you verify that count one more time and explain briefly?",
    }
)

resp2 = cast(
    ModelResponse,
    completion(
        model="openrouter/stealth/ox-alpha",
        messages=messages,
        reasoning={"enabled": True},
        max_tokens=800,
    ),
)

print(f"\n{'━'*60}")
print(f"  Turn 2 — Follow-up answer")
print(f"{'━'*60}")
print(f"[answer]\n{resp2.choices[0].message.content}")


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Provider List: https://docs.litellm.ai/docs/providers


  Turn 1 — First answer
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
There are **3** r's in the word "strawberry."

Breaking it down: s-t-**r**-a-w-b-e-**r**-**r**-y

One "r" appears in the middle ("straw") and two appear near the end ("berry").

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Turn 2 — Follow-up answer
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
Verified — still **3** r's.

Letter-by-letter check of s-t-r-a-w-b-e-r-r-y:

- Position 3: **r** (in "straw")
- Position 8: **r** (in "berry")
- Position 9: **r** (in "berry")

The other seven letters (s, t, a, w, b, e, y) are 

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Suppress LiteLLM's "Provider List: ..." stdout noise
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import litellm
import logging

# Turn off the verbose provider-list prints
litellm.suppress_debug_info = True

# Optional: also quiet the underlying logger
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4 · Streaming with reasoning
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

response = completion(
    model="openrouter/stealth/ox-alpha",
    messages=[
        {"role": "user", "content": "How many r's are in the word 'strawberry'? Explain step by step."}
    ],
    reasoning={"enabled": True},
    stream=True,
    max_tokens=600,
)

print(f"\n{'━'*60}")
print(f"  Streaming response")
print(f"{'━'*60}\n")

for chunk in response:
    # Some providers may yield (event_name, payload) tuples in stream mode.
    if isinstance(chunk, tuple):
        _, chunk = chunk

    chunk = cast(ModelResponse, chunk)
    if not chunk.choices:
        continue

    choice = chunk.choices[0]
    delta = getattr(choice, "delta", None)
    content = getattr(delta, "content", None) if delta else None

    # Fallback for providers that stream message-like chunks
    if not content:
        message = getattr(choice, "message", None)
        content = getattr(message, "content", None) if message else None

    if content:
        print(content, end="", flush=True)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Streaming response
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Counting the R's in "strawberry"

Let me go through the word letter by letter:

**s** — not an r
**t** — not an r
**r** — ✅ 1st r
**a** — not an r
**w** — not an r
**b** — not an r
**e** — not an r
**r** — ✅ 2nd r
**r** — ✅ 3rd r
**y** — not an r

## Answer: There are **3** r's in "strawberry" 🍓

They appear in:
1. "**str**awberry" — after the "st"
2. "strawbe**rr**y" — the double r near the end

In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5 · Helper function — clean Q / answer / reasoning printer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from typing import cast
from litellm import ModelResponse, ModelResponseStream

def ask_ox_alpha(question: str, max_tokens: int = 600):
    resp = completion(
        model="openrouter/stealth/ox-alpha",
        messages=[{"role": "user", "content": question}],
        reasoning={"enabled": True},
        max_tokens=max_tokens,
    )

    # Guard: sometimes a provider may return a tuple or unexpected type
    if isinstance(resp, tuple):
        _, resp = resp

    resp = cast(ModelResponse, resp)

    if not resp.choices:
        print(f"\n{'━'*60}")
        print(f"  Q: {question}")
        print(f"{'━'*60}")
        print("[error] No choices returned")
        return resp

    choice = resp.choices[0]
    msg = getattr(choice, "message", None)

    print(f"\n{'━'*60}")
    print(f"  Q: {question}")
    print(f"{'━'*60}")

    if msg is None:
        print("[error] No message in choice")
        return resp

    # ── Answer ────────────────────────────────────────────────────────
    content = getattr(msg, "content", None)
    print(f"[answer]\n{content if content else '(empty)'}")

    # ── Reasoning (common locations) ──────────────────────────────────
    reasoning = (
        getattr(msg, "reasoning_content", None)
        or getattr(msg, "reasoning", None)
        or (getattr(msg, "provider_specific_fields", {}) or {}).get("reasoning_content")
        or (getattr(msg, "provider_specific_fields", {}) or {}).get("reasoning_details")
    )

    if reasoning:
        print(f"\n[reasoning]\n{reasoning}")
    elif hasattr(msg, "provider_specific_fields") and msg.provider_specific_fields:
        print(f"\n[provider fields]\n{msg.provider_specific_fields}")

    return resp


# ── Test the helper ───────────────────────────────────────────────────
ask_ox_alpha("How many r's are in the word 'strawberry'?")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q: How many r's are in the word 'strawberry'?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[answer]
There are **3** r's in the word "strawberry":

s-t-**r**-a-w-b-e-**r**-**r**-y

One "r" appears after "st" and two more appear near the end ("rr").

[reasoning]
The user is asking how many r's are in the word 'strawberry'.

Let me spell it out: s-t-r-a-w-b-e-r-r-y

Counting the r's:
- s (1st letter)
- t (2nd letter)
- r (3rd letter) - that's one r
- a (4th letter)
- w (5th letter)
- b (6th letter)
- e (7th letter)
- r (8th letter) - that's two
- r (9th letter) - that's three
- y (10th letter)

So there are 3 r's in "strawberry".


ModelResponse(id='gen-1787465503-j8HPNyxTY8ocn2RLSQEd', created=1787465503, model='stealth/ox-alpha', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='There are **3** r\'s in the word "strawberry":\n\ns-t-**r**-a-w-b-e-**r**-**r**-y\n\nOne "r" appears after "st" and two more appear near the end ("rr").', role='assistant', tool_calls=None, function_call=None, reasoning_content='The user is asking how many r\'s are in the word \'strawberry\'.\n\nLet me spell it out: s-t-r-a-w-b-e-r-r-y\n\nCounting the r\'s:\n- s (1st letter)\n- t (2nd letter)\n- r (3rd letter) - that\'s one r\n- a (4th letter)\n- w (5th letter)\n- b (6th letter)\n- e (7th letter)\n- r (8th letter) - that\'s two\n- r (9th letter) - that\'s three\n- y (10th letter)\n\nSo there are 3 r\'s in "strawberry".', provider_specific_fields={'refusal': None, 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking how many r\'s are in the 

In [11]:
## Fallback

In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LiteLLM | Stage 2 — Router + Fallbacks (OpenRouter)
# Priority order:
#   1. anthropic/claude-opus-5-fast
#   2. x-ai/grok-4.6
#   3. meta/muse-spark-1.2-contributor
#   4. stealth/ox-alpha          ← last resort
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os
import litellm
from litellm.router import Router
from typing import cast
from litellm import ModelResponse

# ── quiet the provider-list noise ─────────────────────────────────────
litellm.suppress_debug_info = True


In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1 · Build the Router with ordered deployments + fallbacks
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

router = Router(
    model_list=[
        # Primary
        {
            "model_name": "smart",                          # alias you call
            "litellm_params": {
                "model": "openrouter/anthropic/claude-opus-5-fast",
                "api_key": os.environ["OPENROUTER_API_KEY"],
            },
        },
        # Fallback 1
        {
            "model_name": "smart",
            "litellm_params": {
                "model": "openrouter/x-ai/grok-4.6",
                "api_key": os.environ["OPENROUTER_API_KEY"],
            },
        },
        # Fallback 2
        {
            "model_name": "smart",
            "litellm_params": {
                "model": "openrouter/meta/muse-spark-1.2-contributor",
                "api_key": os.environ["OPENROUTER_API_KEY"],
            },
        },
        # Last resort
        {
            "model_name": "smart",
            "litellm_params": {
                "model": "openrouter/stealth/ox-alpha",
                "api_key": os.environ["OPENROUTER_API_KEY"],
            },
        },
    ],
    # Explicit fallback chain (optional but clear)
    fallbacks=[
        {"smart": ["openrouter/x-ai/grok-4.6",
                   "openrouter/meta/muse-spark-1.2-contributor",
                   "openrouter/stealth/ox-alpha"]}
    ],
    num_retries=2,          # retry each deployment twice before falling over
    timeout=45,
)

print("✓ Router ready")
print("  Order → Claude Opus 5 Fast → Grok 4.6 → Muse Spark → ox-alpha")

11:47:36 - LiteLLM:WARNING: utils.py:2898 - register_model: model=bec2469b156c41ac4b82ced4b63f3e5d599959366c1a65d75b87a94b18c04209 not in built-in cost map and no prefix/region variant matched; cache cost fields will default to 0. To track cache cost, add cache_creation_input_token_cost and cache_read_input_token_cost to model_info
11:47:36 - LiteLLM:WARNING: utils.py:2898 - register_model: model=openrouter/anthropic/claude-opus-5-fast not in built-in cost map and no prefix/region variant matched; cache cost fields will default to 0. To track cache cost, add cache_creation_input_token_cost and cache_read_input_token_cost to model_info
11:47:36 - LiteLLM:WARNING: utils.py:2898 - register_model: model=8fc06f94ab1e8e1d1a0e464d9031ef206dc6d4d955009160b9dac81e705f34aa not in built-in cost map and no prefix/region variant matched; cache cost fields will default to 0. To track cache cost, add cache_creation_input_token_cost and cache_read_input_token_cost to model_info
11:47:36 - LiteLLM:WARN

✓ Router ready
  Order → Claude Opus 5 Fast → Grok 4.6 → Muse Spark → ox-alpha


In [16]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3 · Streaming through the Router
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

stream = router.completion(
    model="smart",
    messages=[{"role": "user", "content": "Hello"}],
    stream=True,
    max_tokens=80,
)


print(f"\n  Streaming response {'━'*20}\n")


for chunk in stream:
    if isinstance(chunk, tuple):
        _, chunk = chunk
    chunk = cast(ModelResponse, chunk)
    if not chunk.choices:
        continue
    delta = getattr(chunk.choices[0], "delta", None)
    content = getattr(delta, "content", None) if delta else None
    if content:
        print(content, end="", flush=True)



  Streaming response ━━━━━━━━━━━━━━━━━━━━

Hello! How's it going? Is there something I can help you with today, or would you just like to chat?

In [19]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Robust streaming through the Router
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

from typing import cast
from litellm import ModelResponse

prompt = "Explain What is model quantization in deep learning and large language models?"

print(f"\n  Streaming response {'━'*20}\n")

try:
    stream = router.completion(
        model="smart",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        max_tokens=400,          # give more room
    )

    got_any_content = False
    model_used = None

    for chunk in stream:
        # Handle possible (event, payload) tuples
        if isinstance(chunk, tuple):
            _, chunk = chunk

        chunk = cast(ModelResponse, chunk)

        # Capture model name if present
        if getattr(chunk, "model", None):
            model_used = chunk.model

        if not chunk.choices:
            continue

        choice = chunk.choices[0]
        delta = getattr(choice, "delta", None)

        # Primary path
        content = getattr(delta, "content", None) if delta else None

        # Fallback paths some providers use
        if not content and delta is not None:
            content = getattr(delta, "text", None)

        if not content:
            message = getattr(choice, "message", None)
            content = getattr(message, "content", None) if message else None

        if content:
            print(content, end="", flush=True)
            got_any_content = True

    print()  # final newline

    if not got_any_content:
        print("\n[warning] Stream finished but no content was received.")
        print("          Trying a non-streaming call to diagnose…")

        # Diagnostic non-streaming call
        resp = router.completion(
            model="smart",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
        )
        if isinstance(resp, tuple):
            _, resp = resp
        resp = cast(ModelResponse, resp)

        print(f"\n  Non-stream model used → {resp.model}")
        print(f"[answer]\n{resp.choices[0].message.content}")
    else:
        if model_used:
            print(f"\n\n  model used → {model_used}")

except Exception as e:
    print(f"\n[error] {type(e).__name__}: {e}")


  Streaming response ━━━━━━━━━━━━━━━━━━━━

**Model quantization reduces the numerical precision of a neural network’s weights (and often activations) from high-precision formats such as FP32 or FP16 to lower-bit representations (INT8, INT4, or mixed/low-bit schemes).** This shrinks memory footprint, lowers bandwidth needs, and speeds inference—especially valuable for large language models (LLMs) with billions of parameters.

### Why it matters
Full-precision models are memory- and compute-heavy. An FP16 7B-parameter LLM already occupies ~14 GB just for weights; larger models quickly become impractical on consumer GPUs or edge devices. Quantization typically yields 2–8× compression (e.g., INT8 ≈ 4× vs FP32; 4-bit ≈ 8×) with only modest accuracy loss when done carefully. Integer arithmetic is also faster and more energy-efficient on modern hardware (GPUs, NPUs, CPUs with SIMD). For LLMs this enables running capable models locally, reducing serving cost, and supporting longer contexts vi

In [24]:
# ! uv pip install "litellm[proxy]"

### **LiteLLM Proxy**

In [38]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

MASTER_KEY = "sk-litellm-local-1234"  
# your proxy's general_settings.master_key

client = OpenAI(
    base_url="http://localhost:4000",
    api_key=MASTER_KEY,
)

# resp = client.chat.completions.create(
#     model="glm-5.3",
#     messages=[{"role": "user", "content": "What is 2+2?"}],
#     max_tokens=500,
# )
# print(resp.choices[0].message.content)

In [39]:
resp = client.chat.completions.create(
    model="max",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=500,
)
print(resp.choices[0].message.content)

2 + 2 = **4**


In [43]:
from openai.types.chat import ChatCompletionMessageParam

messages: list[ChatCompletionMessageParam] = [
    {"role": "user", "content": "How many r's are in the word 'strawberry'?"}
]

resp = client.chat.completions.create(
    model="high",
    messages=messages,
    max_tokens=500,
)
assistant_msg = resp.choices[0].message

messages.append(
    cast(
        ChatCompletionMessageParam,
        {
            "role": "assistant",
            "content": assistant_msg.content or "",
            "reasoning_details": getattr(assistant_msg, "reasoning_details", None),
        },
    )
)
messages.append({"role": "user", "content": "Are you sure? Think carefully."})

resp2 = client.chat.completions.create(
    model="high",
    messages=messages,
    max_tokens=500,
)
print(resp2.choices[0].message.content)

Yes, I'm confident. Let me verify by spelling it out letter by letter:

**S-T-R-A-W-B-E-R-R-Y**

1. S — no
2. T — no
3. **R** — yes (1)
4. A — no
5. W — no
6. B — no
7. E — no
8. **R** — yes (2)
9. **R** — yes (3)
10. Y — no

The answer remains **3 r's**: one in "straw" and two in "berry".
